# Web Scraping-Assignment 3

# Q1. Write a python program which searches all the product under a particular product from www.amazon.in. The product to be searched will be taken as input from user. For e.g. If user input is ‘guitar’. Then search for guitars

In [1]:
! pip install selenium

In [2]:
#import all the required libraries
import selenium
import pandas as pd
from selenium import webdriver
import time
from bs4 import BeautifulSoup

#importing exception
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException

In [3]:
# Importing selenium webdriver 
from selenium import webdriver

In [4]:
driver=webdriver.Chrome("chromedriver.exe")

<ipython-input-4-13c3cc54b860>:1: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver=webdriver.Chrome("chromedriver.exe")


In [33]:
# Opening the homepage of Amazon.in
driver.get('https://www.amazon.in/')
# Asking the user to input the keywords he/she wants to search
user_inp = input('Enter the product you want to search : ')

Enter the product you want to search : kurtis


In [34]:
search_bar = driver.find_element_by_id("twotabsearchtextbox")    # Locating searc_bar by id
search_bar.clear()                                               # clearing search_bar
search_bar.send_keys(user_inp)                                   # sending user input to search bar
search_button = driver.find_element_by_xpath('//div[@class="nav-search-submit nav-sprite"]/span/input')       # Locating search_button by xpath
search_button.click()                                                                # Clicking the button to start search

<ipython-input-34-5fc1fd69cac4>:1: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  search_bar = driver.find_element_by_id("twotabsearchtextbox")    # Locating searc_bar by id
<ipython-input-34-5fc1fd69cac4>:4: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  search_button = driver.find_element_by_xpath('//div[@class="nav-search-submit nav-sprite"]/span/input')       # Locating search_button by xpath


# Q2. In the above question, now scrape the following details of each product listed in first 3 pages of your search results and save it in a data frame and csv. In case if any product has less than 3 pages in search results then scrape all the products available under that product name. Details to be scraped are: "Brand Name", "Name of the Product", "Price", "Return/Exchange", "Expected Delivery", "Availability" and “Product URL”. In case, if any of the details are missing for any of the product then replace it by “-

In [35]:
start_page = 0
end_page = 3
urls = []
for page in range(start_page,end_page+1):
    try:
        page_urls = driver.find_elements_by_xpath('//a[@class="a-link-normal s-no-outline"]')
        
        # appending all the urls on current page to urls list
        for url in page_urls:
            url = url.get_attribute('href')     # Scraping the url from webelement
            if url[0:4]=='http':                # Checking if the scraped data is a valid url or not
                urls.append(url)                # Appending the url to urls list
        print("Product urls of page {} has been scraped.".format(page+1))
        
        # Moving to next page
        nxt_button = driver.find_element_by_xpath('//li[@class="a-last"]/a')      # Locating the next_button which is active
        if nxt_button.text == 'Next→':                                            # Checking if the button located is next button
            nxt_button.click()                                                    # Clicking the next button
            time.sleep(5)                                                         # time delay of 5 seconds
        # If the current active button is not next button, the we will check if the next button is inactive or not    
        elif driver.find_element_by_xpath('//li[@class="a-disabled a-last"]/a').text == 'Next→':    
            print("No new pages exist. Breaking the loop")  # Printing message and breakinf loop if we have reached the last page
            break
            
    except StaleElementReferenceException as e:             # Handling StaleElement Exception   
        print("Stale Exception")
        next_page = nxt_button.get_attribute('href')        # Extracting the url of next page
        driver.get(next_page)                               # ReLoading the next page

<ipython-input-35-981f44ac24f7>:6: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  page_urls = driver.find_elements_by_xpath('//a[@class="a-link-normal s-no-outline"]')


Product urls of page 1 has been scraped.


<ipython-input-35-981f44ac24f7>:16: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  nxt_button = driver.find_element_by_xpath('//li[@class="a-last"]/a')      # Locating the next_button which is active


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//li[@class="a-last"]/a"}
  (Session info: chrome=101.0.4951.67)
Stacktrace:
Backtrace:
	Ordinal0 [0x00EB7413+2389011]
	Ordinal0 [0x00E49F61+1941345]
	Ordinal0 [0x00D3C658+837208]
	Ordinal0 [0x00D691DD+1020381]
	Ordinal0 [0x00D6949B+1021083]
	Ordinal0 [0x00D96032+1204274]
	Ordinal0 [0x00D84194+1130900]
	Ordinal0 [0x00D94302+1196802]
	Ordinal0 [0x00D83F66+1130342]
	Ordinal0 [0x00D5E546+976198]
	Ordinal0 [0x00D5F456+980054]
	GetHandleVerifier [0x01069632+1727522]
	GetHandleVerifier [0x0111BA4D+2457661]
	GetHandleVerifier [0x00F4EB81+569713]
	GetHandleVerifier [0x00F4DD76+566118]
	Ordinal0 [0x00E50B2B+1968939]
	Ordinal0 [0x00E55988+1989000]
	Ordinal0 [0x00E55A75+1989237]
	Ordinal0 [0x00E5ECB1+2026673]
	BaseThreadInitThunk [0x7755FA29+25]
	RtlGetAppContainerNamedObjectPath [0x77E97A7E+286]
	RtlGetAppContainerNamedObjectPath [0x77E97A4E+238]


In [36]:
prod_dict = {}
prod_dict['Brand']=[]
prod_dict['Name']=[]
prod_dict['Rating']=[]
prod_dict['No. of ratings']=[]
prod_dict['Price']=[]
prod_dict['Return/Exchange']=[]
prod_dict['Expected Delivery']=[] 
prod_dict['Availability']=[]
prod_dict['Other Details']=[]
prod_dict['URL']=[]

In [37]:
for url in urls[:4]:
    driver.get(url)                                                        # Loading the webpage by url
    print("Scraping URL = ", url)
    #time.sleep(2)
    
    try:
        brand = driver.find_element_by_xpath('//a[@id="bylineInfo"]')      # Extracting Brand from xpath
        prod_dict['Brand'].append(brand.text)
    except NoSuchElementException:
        prod_dict['Brand'].append('-')
    
    try:
        name = driver.find_element_by_xpath('//h1[@id="title"]/span')      # Extracting Name from xpath
        prod_dict['Name'].append(name.text)
    except NoSuchElementException:
        prod_dict['Name'].append('-')
    
    try:
        rating = driver.find_element_by_xpath('//span[@id="acrPopover"]')  # Extracting Ratings from xpath
        prod_dict['Rating'].append(rating.get_attribute("title"))
    except NoSuchElementException:
        prod_dict['Rating'].append('-')
    
    try:
        n_rating = driver.find_element_by_xpath('//a[@id="acrCustomerReviewLink"]/span')     # Extracting no. of Ratings from xpath
        prod_dict['No. of ratings'].append(n_rating.text)
    except NoSuchElementException:
        prod_dict['No. of ratings'].append('-')
    
    try:
        price = driver.find_element_by_xpath('//span[@id="priceblock_ourprice"]')            # Extracting Price from xpath
        prod_dict['Price'].append(price.text)
    except NoSuchElementException:
        prod_dict['Price'].append('-')
    try:                                                                                     # Extracting Return/Exchange policy from xpath
        ret = driver.find_element_by_xpath('//div[@data-name="RETURNS_POLICY"]/span/div[2]/a')
        prod_dict['Return/Exchange'].append(ret.text)
    except NoSuchElementException:
        prod_dict['Return/Exchange'].append('-')
    try:
        delivry = driver.find_element_by_xpath('//div[@id="ddmDeliveryMessage"]/b')         # Extracting Expected Delivery from xpath
        prod_dict['Expected Delivery'].append(delivry.text)
    except NoSuchElementException:
        prod_dict['Expected Delivery'].append('-')
    
    try:
        avl = driver.find_element_by_xpath('//div[@id="availability"]/span')                # Extracting Availability from xpath
        prod_dict['Availability'].append(avl.text)
    except NoSuchElementException:
        prod_dict['Availability'].append('-')
    
    try:                                                                                    # Extracting Other Details from xpath
        dtls = driver.find_element_by_xpath('//ul[@class="a-unordered-list a-vertical a-spacing-mini"]')
        prod_dict['Other Details'].append('  ||  '.join(dtls.text.split('\n')))
    except NoSuchElementException:
        prod_dict['Other Details'].append('-')
    
    prod_dict['URL'].append(url)                                                            # Saving url
    time.sleep(2)

Scraping URL =  https://www.amazon.in/gp/slredirect/picassoRedirect.html/ref=pa_sp_atf_aps_sr_pg1_1?ie=UTF8&adId=A0732279PHILCLOATT4B&url=%2FPort-Bliss-Womens-Straight-Printed_Kurti__Sky%2Fdp%2FB099NNB37T%2Fref%3Dsr_1_1_sspa%3Fkeywords%3Dkurtis%26qid%3D1653156870%26sr%3D8-1-spons%26psc%3D1&qualifier=1653156870&id=2505251558843953&widgetName=sp_atf


<ipython-input-37-5c499602e9be>:7: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  brand = driver.find_element_by_xpath('//a[@id="bylineInfo"]')      # Extracting Brand from xpath
<ipython-input-37-5c499602e9be>:13: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  name = driver.find_element_by_xpath('//h1[@id="title"]/span')      # Extracting Name from xpath
<ipython-input-37-5c499602e9be>:19: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  rating = driver.find_element_by_xpath('//span[@id="acrPopover"]')  # Extracting Ratings from xpath
<ipython-input-37-5c499602e9be>:25: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  n_rating = driver.find_element_by_xpath('//a[@id="acrCustomerReviewLink"]/span')     # Extracting

Scraping URL =  https://www.amazon.in/gp/slredirect/picassoRedirect.html/ref=pa_sp_atf_aps_sr_pg1_1?ie=UTF8&adId=A019114034KIPWNEZ1THA&url=%2FYash-Gallery-Womens-Printed-Anarkali%2Fdp%2FB09QX8TF8G%2Fref%3Dsr_1_2_sspa%3Fkeywords%3Dkurtis%26qid%3D1653156870%26sr%3D8-2-spons%26psc%3D1&qualifier=1653156870&id=2505251558843953&widgetName=sp_atf
Scraping URL =  https://www.amazon.in/gp/slredirect/picassoRedirect.html/ref=pa_sp_atf_aps_sr_pg1_1?ie=UTF8&adId=A0511550248ZQZWEM7UQ8&url=%2FAmazon-Brand-Womens-SS20MYXCORE22_Maroon-Bandhej_Small%2Fdp%2FB083FWHHLQ%2Fref%3Dsr_1_3_sspa%3Fkeywords%3Dkurtis%26qid%3D1653156870%26sr%3D8-3-spons%26psc%3D1&qualifier=1653156870&id=2505251558843953&widgetName=sp_atf
Scraping URL =  https://www.amazon.in/gp/slredirect/picassoRedirect.html/ref=pa_sp_atf_aps_sr_pg1_1?ie=UTF8&adId=A081596621E21ZE2SIN0E&url=%2FPALASHBONI-Women-Cotton-Kurti-Ethnic%2Fdp%2FB09XSRH19D%2Fref%3Dsr_1_4_sspa%3Fkeywords%3Dkurtis%26qid%3D1653156870%26sr%3D8-4-spons%26psc%3D1&qualifier=16531

In [38]:
prod_df = pd.DataFrame.from_dict(prod_dict)
prod_df

,Brand,Name,Rating,No. of ratings,Price,Return/Exchange,Expected Delivery,Availability,Other Details,URL
0,Brand: Port Bliss,Port Bliss Women's Pure Cotton Printed Straigh...,-,-,-,-,-,Currently unavailable.,Care Instructions: Hand Wash Only || Port Bl...,https://www.amazon.in/gp/slredirect/picassoRed...
1,Visit the Yash Gallery Store,Yash Gallery Women's Floral Printed Anarkali K...,4.8 out of 5 stars,8 ratings,-,30 Days Returns & Exchange,-,In stock.,Care Instructions: Hand Wash Only || Fit Typ...,https://www.amazon.in/gp/slredirect/picassoRed...
2,Visit the Amazon Brand - Myx Store,Amazon Brand - Myx Women's Cotton Kurti,4.0 out of 5 stars,"5,575 ratings",-,30 Days Returns & Exchange,-,In stock.,Care Instructions: Machine Wash || Fit Type:...,https://www.amazon.in/gp/slredirect/picassoRed...
3,Brand: Generic,PALASHBONI Women Cotton Kurti Ethnic-,-,-,-,30 Days Returns & Exchange,-,Only 1 left in stock.,Care Instructions: Machine Wash || PURE COTT...,https://www.amazon.in/gp/slredirect/picassoRed...


In [39]:
#saving data to csv
prod_df.to_csv('Amazon_{}.csv'.format(user_inp))

# Q.3. Write a python program to access the search bar and search button on images.google.com and scrape 10 images each for keywords ‘fruits’, ‘cars’ and ‘Machine Learning’, ‘Guitar’, ‘Cakes’.

In [84]:
#Activating the chrome browser with specified url
driver = webdriver.Chrome("chromedriver.exe")
# getting images.google.com
url = "https://images.google.com/"
#Creating empty list and giving search items as list and creating loop
urls = []    
data = []
search_item = ["fruits", "cars", "Machine Learning"]
for item in search_item:
    driver.get(url)  
    time.sleep(5)
    search_bar = driver.find_element_by_tag_name("input") #Xpath for search bar
    
    search_bar.send_keys(str(item))      #sending key word for search item
    
    search_button =driver.find_element_by_xpath("//button[@class='Tg7LZd']").click() #Clicking on search button
    
    # scrolling the web page to get more images
    for _ in range(500):
        driver.execute_script("window.scrollBy(0,100)")
        
        imgs = driver.find_elements_by_xpath("//img[@class='rg_i Q4LuWd']")
    img_url = []
    for image in imgs:
        source = image.get_attribute('src')
        if source is not None:
                if(source[0:4] == 'http'):
                    img_url.append(source)
    for i in img_url[:100]:
        urls.append(i)
                    
for i in range(len(urls)):
    if i >= 300:
        break
    print("Downloading {0} of {1} images" .format(i, 300))
    response = requests.get(urls[i])

    file = open(r"F:\images"+str(i)+".jpg", "wb")

    file.write(response.content)

<ipython-input-84-2dee2f321a4b>:2: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome("chromedriver.exe")
<ipython-input-84-2dee2f321a4b>:12: DeprecationWarning: find_element_by_tag_name is deprecated. Please use find_element(by=By.TAG_NAME, value=name) instead
  search_bar = driver.find_element_by_tag_name("input") #Xpath for search bar
<ipython-input-84-2dee2f321a4b>:16: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  search_button =driver.find_element_by_xpath("//button[@class='Tg7LZd']").click() #Clicking on search button
<ipython-input-84-2dee2f321a4b>:22: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  imgs = driver.find_elements_by_xpath("//img[@class='rg_i Q4LuWd']")


NameError: name 'requests' is not defined

# Q4. Write a python program to search for a smartphone(e.g.: Oneplus Nord, pixel 4A, etc.) on www.flipkart.com and scrape following details for all the search results displayed on 1st page. Details to be scraped: “Brand Name”, “Smartphone name”, “Colour”, “RAM”, “Storage(ROM)”, “Primary Camera”, “Secondary Camera”, “Display Size”, “Battery Capacity”, “Price”, “Product URL”. Incase if any of the details is missing then replace it by “- “. Save your results in a dataframe and CSV.

In [86]:
# Activating the chrome browser
item = input(" Enter the name of Smartphone that has to be searched : ")
driver = webdriver.Chrome("chromedriver.exe") 

#get the web page with given url
url = "https://www.flipkart.com/"
driver.get(url)
time.sleep(3)

 Enter the name of Smartphone that has to be searched : Pixel


<ipython-input-86-71cf01604809>:3: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome("chromedriver.exe")


In [87]:
#Clicking on login close icon
login_X_btn = driver.find_element_by_xpath("//div[@class='_2QfC02']//button").click()

<ipython-input-87-0a60051ad0a2>:2: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  login_X_btn = driver.find_element_by_xpath("//div[@class='_2QfC02']//button").click()


In [88]:
#giving input key word to search bar
serch_bar = driver.find_element_by_xpath("//div[@class='_3OO5Xc']//input")
serch_bar.send_keys(item)

srch_btn = driver.find_element_by_xpath("/html/body/div[1]/div/div[1]/div[1]/div[2]/div[2]/form/div/button")
srch_btn.click()
time.sleep(5)# Fetching urls of phones coming on 1st page
flip_urls = []
urls = driver.find_elements_by_xpath('//a[@class="_1fQZEK"]')
for url in urls:
    flip_urls.append(url.get_attribute("href"))

<ipython-input-88-f74a12ccb113>:2: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  serch_bar = driver.find_element_by_xpath("//div[@class='_3OO5Xc']//input")
<ipython-input-88-f74a12ccb113>:5: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  srch_btn = driver.find_element_by_xpath("/html/body/div[1]/div/div[1]/div[1]/div[2]/div[2]/form/div/button")
<ipython-input-88-f74a12ccb113>:9: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  urls = driver.find_elements_by_xpath('//a[@class="_1fQZEK"]')


In [89]:
# Fetching urls of phones coming on 1st page
page1_urls = []
urls = driver.find_elements_by_xpath('//a[@class="_1fQZEK"]')
for url in urls:
    page1_urls.append(url.get_attribute("href"))
    
len(page1_urls)

<ipython-input-89-9332b4f0666d>:3: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  urls = driver.find_elements_by_xpath('//a[@class="_1fQZEK"]')


24

In [90]:
Smartphones = {}
Smartphones["Brand"] = []
Smartphones["Phone name"] = []
Smartphones["Colour"] = []
Smartphones["RAM"] = []
Smartphones["Storage(ROM)"] = []
Smartphones["Primary Camera"] = []
Smartphones["Secondary Camera"] = []
Smartphones["Display Size"] = []
Smartphones["Display Resolution"] = []
Smartphones["Processor"] = []
Smartphones["Processor Cores"] = []
Smartphones["Battery Capacity"] = []
Smartphones["Price"] = []
Smartphones["URL"] = []

In [91]:
# Scraping data from each url of page 1
for url in page1_urls:
    driver.get(url)                                                        
    print("Scraping URL = ", url)
    Smartphones['URL'].append(url)                                                          
    time.sleep(2)
    
    
    #Clicking on read more button
    try:
        read_more = driver.find_element_by_xpath('//button[@class="_2KpZ6l _1FH0tX"]')     
        read_more.click()
    except NoSuchElementException:
        print("Exception occured while moving to next page")
    
    
    #Scraping brand name of phone data
    try:
        brand_tags = driver.find_element_by_xpath('//span[@class="B_NuCI"]')      
        Smartphones["Brand"].append(brand_tags.text.split()[0])
    except NoSuchElementException:
        Smartphones['Brand'].append('-')
    
    
    #Scraping phone name data
    try:
        name_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][1]/table/tbody/tr[3]/td[2]/ul/li')     
        Smartphones['Phone name'].append(name_tags.text)
    except NoSuchElementException:
        Smartphones['Phone name'].append('-')
    
    
    #Scraping phone color data
    try:
        color_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][1]/table/tbody/tr[4]/td[2]/ul/li')      
        Smartphones['Colour'].append(color_tags.text)
    except NoSuchElementException:
        Smartphones['Colour'].append('-')
     
    
    #Scraping RAM data
    try:
        ram_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][4]/table[1]/tbody/tr[2]/td[2]/ul/li')                
        Smartphones['RAM'].append(ram_tags.text)
    except NoSuchElementException:
        Smartphones['RAM'].append('-')
    
    
    #Scraping ROM data
    try:
        rom_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][4]/table[1]/tbody/tr[1]/td[2]/ul/li')        
        Smartphones['Storage(ROM)'].append(rom_tags.text)
    except NoSuchElementException:
        Smartphones['Storage(ROM)'].append('-')
        
        
    #Scraping Primary camera data
    try:                                                                                    
        pri_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][5]/table[1]/tbody/tr[2]/td[2]/ul/li')
        Smartphones['Primary Camera'].append(pri_tags.text)
    except NoSuchElementException:
        Smartphones['Primary Camera'].append('-')
        
        
    #Scraping secondary camera data
    try:                                                                                    
        sec_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][5]/table[1]/tbody/tr[6]/td[1]')
        if sec_tags != "Secondary Camera" : 
            if driver.find_element_by_xpath('//div[@class="_3k-BhJ"][5]/table[1]/tbody/tr[5]/td[1]').text == "Secondary Camera":
                sec_cam = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][5]/table[1]/tbody/tr[5]/td[2]/ul/li')
            else :
                raise NoSuchElementException
        else :
            sec_cam = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][5]/table[1]/tbody/tr[6]/td[2]/ul/li')
        Smartphones['Secondary Camera'].append(sec_cam.text)
    except NoSuchElementException:
        Smartphones['Secondary Camera'].append('-')
        
        
    #Scraping Display size data 
    try:
        disp_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][2]/div')
        if disp_tags.text != "Display Features" : raise NoSuchElementException
        disp_size = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][2]/table[1]/tbody/tr[1]/td[2]/ul/li')  
        Smartphones['Display Size'].append(disp_size.text)
    except NoSuchElementException:
        Smartphones['Display Size'].append('-')
    
    
    #Scraping display resolution data
    try:
        dires_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][2]/div')
        if dires_tags.text != "Display Features" : raise NoSuchElementException
        disp_reso = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][2]/table[1]/tbody/tr[2]/td[2]/ul/li')    
        Smartphones['Display Resolution'].append(disp_reso.text)
    except NoSuchElementException:
        Smartphones['Display Resolution'].append('-')
    
    
    
    #Scraping Processor data
    try:
        pro_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][3]/table[1]/tbody/tr[2]/td[1]')
        if pro_tags.text != "Processor Type" : raise NoSuchElementException
        processor = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][3]/table[1]/tbody/tr[2]/td[2]/ul/li')   
        Smartphones['Processor'].append(processor.text)
    except NoSuchElementException:
        Smartphones['Processor'].append('-')
        
    
    
    #Scraping Processor core data    
    try:                                                                                     
        core_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][3]/table[1]/tbody/tr[3]/td[1]')
        if core_tags.text != "Processor Core" :
            core_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][3]/table[1]/tbody/tr[2]/td[1]')
            if core_tags.text != "Processor Core" : 
                raise NoSuchElementException
            else :
                cores = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][3]/table[1]/tbody/tr[2]/td[2]/ul/li')
        else :
            cores = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][3]/table[1]/tbody/tr[3]/td[2]/ul/li')
        Smartphones['Processor Cores'].append(cores.text)
    except NoSuchElementException:
        Smartphones['Processor Cores'].append('-')
    
    
    
    #Scraping battery capacity data
    try:
        if driver.find_element_by_xpath('//div[@class="_3k-BhJ"][10]/div').text != "Battery & Power Features" :
            if driver.find_element_by_xpath('//div[@class="_3k-BhJ"][9]/div').text == "Battery & Power Features" :
                bat_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][9]/table/tbody/tr/td[1]')
                if bat_tags.text != "Battery Capacity" : raise NoSuchElementException
                bat_capa = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][9]/table/tbody/tr/td[2]/ul/li')                
            elif driver.find_element_by_xpath('//div[@class="_3k-BhJ"][8]/div').text == "Battery & Power Features" :
                bat_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][8]/table/tbody/tr/td[1]')
                if bat_tags.text != "Battery Capacity" : raise NoSuchElementException
                bat_capa = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][8]/table/tbody/tr/td[2]/ul/li')
            else:
                raise NoSuchElementException
        else :
            bat_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][10]/table/tbody/tr/td[1]')
            if bat_tags.text != "Battery Capacity" : raise NoSuchElementException
            bat_capa = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][10]/table/tbody/tr/td[2]/ul/li')              
        Smartphones['Battery Capacity'].append(bat_capa.text)
    except NoSuchElementException:
        Smartphones['Battery Capacity'].append('-')
        
        
        
    #Scraping Price data
    try:
        price_tags = driver.find_element_by_xpath('//div[@class="_30jeq3 _16Jk6d"]')      
        Smartphones['Price'].append(price_tags.text)
    except NoSuchElementException:
        Smartphones['Price'].append('-')

Scraping URL =  https://www.flipkart.com/google-pixel-4a-just-black-128-gb/p/itm023b9677aa45d?pid=MOBFUSBNAZGY7HQU&lid=LSTMOBFUSBNAZGY7HQUWHTF0C&marketplace=FLIPKART&q=Pixel&store=tyy%2F4io&srno=s_1_1&otracker=search&otracker1=search&fm=organic&iid=da55827b-0dcb-4a53-a40e-7aab3a4cd4c1.MOBFUSBNAZGY7HQU.SEARCH&ppt=hp&ppn=homepage&ssid=95ucn8n92o0000001653323907709&qH=08822b3ae4e2aede


<ipython-input-91-07536367431e>:11: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  read_more = driver.find_element_by_xpath('//button[@class="_2KpZ6l _1FH0tX"]')
<ipython-input-91-07536367431e>:19: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  brand_tags = driver.find_element_by_xpath('//span[@class="B_NuCI"]')
<ipython-input-91-07536367431e>:27: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  name_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][1]/table/tbody/tr[3]/td[2]/ul/li')
<ipython-input-91-07536367431e>:35: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  color_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][1]/table/tbody/tr[4]/td[2]/ul/li')
<ipython-input-91-07536367431e>:43: D

Scraping URL =  https://www.flipkart.com/google-pixel-3a-clearly-white-64-gb/p/itmfgk4jfgstaack?pid=MOBFFGFPJSCEXMSG&lid=LSTMOBFFGFPJSCEXMSGODGRZE&marketplace=FLIPKART&q=Pixel&store=tyy%2F4io&srno=s_1_2&otracker=search&otracker1=search&fm=organic&iid=da55827b-0dcb-4a53-a40e-7aab3a4cd4c1.MOBFFGFPJSCEXMSG.SEARCH&ppt=hp&ppn=homepage&ssid=95ucn8n92o0000001653323907709&qH=08822b3ae4e2aede


<ipython-input-91-07536367431e>:136: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  elif driver.find_element_by_xpath('//div[@class="_3k-BhJ"][8]/div').text == "Battery & Power Features" :
<ipython-input-91-07536367431e>:137: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  bat_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][8]/table/tbody/tr/td[1]')
<ipython-input-91-07536367431e>:139: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  bat_capa = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][8]/table/tbody/tr/td[2]/ul/li')


Scraping URL =  https://www.flipkart.com/vivo-t1-44w-midnight-galaxy-128-gb/p/itm6fabe8894256b?pid=MOBGDRHVXFVCGS23&lid=LSTMOBGDRHVXFVCGS23RDLBPG&marketplace=FLIPKART&q=Pixel&store=tyy%2F4io&srno=s_1_3&otracker=search&otracker1=search&fm=organic&iid=da55827b-0dcb-4a53-a40e-7aab3a4cd4c1.MOBGDRHVXFVCGS23.SEARCH&ppt=hp&ppn=homepage&ssid=95ucn8n92o0000001653323907709&qH=08822b3ae4e2aede


<ipython-input-91-07536367431e>:143: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  bat_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][10]/table/tbody/tr/td[1]')
<ipython-input-91-07536367431e>:145: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  bat_capa = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][10]/table/tbody/tr/td[2]/ul/li')


Scraping URL =  https://www.flipkart.com/vivo-t1-44w-starry-sky-128-gb/p/itm6fabe8894256b?pid=MOBGDRHVHNBBBBP5&lid=LSTMOBGDRHVHNBBBBP57SXRTY&marketplace=FLIPKART&q=Pixel&store=tyy%2F4io&srno=s_1_4&otracker=search&otracker1=search&fm=organic&iid=da55827b-0dcb-4a53-a40e-7aab3a4cd4c1.MOBGDRHVHNBBBBP5.SEARCH&ppt=hp&ppn=homepage&ssid=95ucn8n92o0000001653323907709&qH=08822b3ae4e2aede
Scraping URL =  https://www.flipkart.com/vivo-t1-44w-ice-dawn-128-gb/p/itm6fabe8894256b?pid=MOBGDRHV86NZHMDM&lid=LSTMOBGDRHV86NZHMDMR0BLSS&marketplace=FLIPKART&q=Pixel&store=tyy%2F4io&srno=s_1_5&otracker=search&otracker1=search&fm=organic&iid=da55827b-0dcb-4a53-a40e-7aab3a4cd4c1.MOBGDRHV86NZHMDM.SEARCH&ppt=hp&ppn=homepage&ssid=95ucn8n92o0000001653323907709&qH=08822b3ae4e2aede
Scraping URL =  https://www.flipkart.com/vivo-t1-44w-midnight-galaxy-128-gb/p/itm6fabe8894256b?pid=MOBGDRHVZN29ZJF4&lid=LSTMOBGDRHVZN29ZJF4QSKOSJ&marketplace=FLIPKART&q=Pixel&store=tyy%2F4io&srno=s_1_6&otracker=search&otracker1=search&fm=or

<ipython-input-91-07536367431e>:116: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  core_tags = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][3]/table[1]/tbody/tr[2]/td[1]')
<ipython-input-91-07536367431e>:120: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  cores = driver.find_element_by_xpath('//div[@class="_3k-BhJ"][3]/table[1]/tbody/tr[2]/td[2]/ul/li')


Scraping URL =  https://www.flipkart.com/infinix-smart-6-starry-purple-64-gb/p/itmd4041ce33d262?pid=MOBGDC6HFGCW5PGF&lid=LSTMOBGDC6HFGCW5PGFFKFQIM&marketplace=FLIPKART&q=Pixel&store=tyy%2F4io&srno=s_1_11&otracker=search&otracker1=search&fm=organic&iid=da55827b-0dcb-4a53-a40e-7aab3a4cd4c1.MOBGDC6HFGCW5PGF.SEARCH&ppt=hp&ppn=homepage&ssid=95ucn8n92o0000001653323907709&qH=08822b3ae4e2aede
Scraping URL =  https://www.flipkart.com/vivo-t1-pro-5g-turbo-cyan-128-gb/p/itmdbb5d58c3018d?pid=MOBGDRHVYUZFQEDE&lid=LSTMOBGDRHVYUZFQEDEL4F8JS&marketplace=FLIPKART&q=Pixel&store=tyy%2F4io&srno=s_1_12&otracker=search&otracker1=search&fm=organic&iid=da55827b-0dcb-4a53-a40e-7aab3a4cd4c1.MOBGDRHVYUZFQEDE.SEARCH&ppt=hp&ppn=homepage&ssid=95ucn8n92o0000001653323907709&qH=08822b3ae4e2aede
Scraping URL =  https://www.flipkart.com/redmi-10a-slate-grey-64-gb/p/itm755830dd54ae1?pid=MOBGDRGV2UAGZYZG&lid=LSTMOBGDRGV2UAGZYZGOV9P34&marketplace=FLIPKART&q=Pixel&store=tyy%2F4io&srno=s_1_13&otracker=search&otracker1=search

In [92]:
#Checking lengths of all scraped data
print(len(Smartphones["Brand"]), len(Smartphones["Phone name"]), len(Smartphones["Colour"]), len(Smartphones["RAM"]), len(Smartphones["Storage(ROM)"]), len(Smartphones["Primary Camera"]), len(Smartphones["Secondary Camera"]), len(Smartphones["Display Size"]), len(Smartphones["Display Resolution"]), len(Smartphones["Processor"]), len(Smartphones["Processor Cores"]), len(Smartphones["Battery Capacity"]), len(Smartphones["Price"]), len(Smartphones['URL']))

24 24 24 24 24 24 24 24 24 24 24 24 24 24


In [93]:
#Framing the data
df = pd.DataFrame.from_dict(Smartphones)
df

,Brand,Phone name,Colour,RAM,Storage(ROM),Primary Camera,Secondary Camera,Display Size,Display Resolution,Processor,Processor Cores,Battery Capacity,Price,URL
0,Google,Pixel 4a,Just Black,6 GB,128 GB,12.2MP Rear Camera,8MP Front Camera,14.76 cm (5.81 inch),2340 x 1080 Pixels,Qualcomm Snapdragon 730G,Octa Core,3140 mAh,"₹31,999",https://www.flipkart.com/google-pixel-4a-just-...
1,Google,Pixel 3a,Clearly White,4 GB,64 GB,12.2MP Rear Camera,8MP Front Camera,14.22 cm (5.6 inch),2220 x 1080 pixels,Qualcomm Snapdragon 670,Octa Core,3000 mAh,"₹39,749",https://www.flipkart.com/google-pixel-3a-clear...
2,vivo,T1 44W,Midnight Galaxy,6 GB,128 GB,50MP + 2MP + 2MP,16MP Front Camera,16.36 cm (6.44 inch),2400 x 1080 Pixel,Qualcomm Snapdragon 680,Octa Core,5000 mAh,"₹15,999",https://www.flipkart.com/vivo-t1-44w-midnight-...
3,vivo,T1 44W,Starry Sky,4 GB,128 GB,50MP + 2MP + 2MP,16MP Front Camera,16.36 cm (6.44 inch),2400 x 1080 Pixel,Qualcomm Snapdragon 680,Octa Core,5000 mAh,"₹14,499",https://www.flipkart.com/vivo-t1-44w-starry-sk...
4,vivo,T1 44W,Ice Dawn,8 GB,128 GB,50MP + 2MP + 2MP,16MP Front Camera,16.36 cm (6.44 inch),2400 x 1080 Pixel,Qualcomm Snapdragon 680,Octa Core,5000 mAh,"₹17,999",https://www.flipkart.com/vivo-t1-44w-ice-dawn-...
5,vivo,T1 44W,Midnight Galaxy,4 GB,128 GB,50MP + 2MP + 2MP,16MP Front Camera,16.36 cm (6.44 inch),2400 x 1080 Pixel,Qualcomm Snapdragon 680,Octa Core,5000 mAh,"₹14,499",https://www.flipkart.com/vivo-t1-44w-midnight-...
6,vivo,T1 44W,Starry Sky,8 GB,128 GB,50MP + 2MP + 2MP,16MP Front Camera,16.36 cm (6.44 inch),2400 x 1080 Pixel,Qualcomm Snapdragon 680,Octa Core,5000 mAh,"₹17,999",https://www.flipkart.com/vivo-t1-44w-starry-sk...
7,vivo,T1 44W,Ice Dawn,4 GB,128 GB,50MP + 2MP + 2MP,16MP Front Camera,16.36 cm (6.44 inch),2400 x 1080 Pixel,Qualcomm Snapdragon 680,Octa Core,5000 mAh,"₹14,499",https://www.flipkart.com/vivo-t1-44w-ice-dawn-...
8,vivo,T1 44W,Midnight Galaxy,8 GB,128 GB,50MP + 2MP + 2MP,16MP Front Camera,16.36 cm (6.44 inch),2400 x 1080 Pixel,Qualcomm Snapdragon 680,Octa Core,5000 mAh,"₹17,999",https://www.flipkart.com/vivo-t1-44w-midnight-...
9,REDMI,REDMI 10A,Sea Blue,4 GB,64 GB,Primary Camera,-,16.59 cm (6.53 inch),1600 x 700$$pixel,-,Octa Core,-,"₹9,982",https://www.flipkart.com/redmi-10a-sea-blue-64...


In [94]:
#saving data to csv
df.to_csv('Flipkart_{}.csv'.format(item))

# Q5. Write a program to scrap geospatial coordinates (latitude, longitude) of a city searched on google maps.

In [36]:
# opening google maps
driver.get("https://www.google.co.in/maps")
time.sleep(3)

city = input('Enter City Name : ')                                         # Enter city to be searched
search = driver.find_element_by_id("searchboxinput")                       # locating search bar
search.clear()                                                             # clearing search bar
time.sleep(2)
search.send_keys(city)                                                     # entering values in search bar
button = driver.find_element_by_id("searchbox-searchbutton")               # locating search button
button.click()                                                             # clicking search button
time.sleep(3)

try:
    url_string = driver.current_url
    print("URL Extracted: ", url_string)
    lat_lng = re.findall(r'@(.*)data',url_string)
    if len(lat_lng):
        lat_lng_list = lat_lng[0].split(",")
        if len(lat_lng_list)>=2:
            lat = lat_lng_list[0]
            lng = lat_lng_list[1]
        print("Latitude = {}, Longitude = {}".format(lat, lng))

except Exception as e:
        print("Error: ", str(e))

Enter City Name : Delhi


<ipython-input-36-efd1c7f00fa3>:6: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  search = driver.find_element_by_id("searchboxinput")                       # locating search bar
<ipython-input-36-efd1c7f00fa3>:10: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  button = driver.find_element_by_id("searchbox-searchbutton")               # locating search button


URL Extracted:  https://www.google.co.in/maps/place/Delhi/@28.6466772,76.8130624,10z/data=!3m1!4b1!4m5!3m4!1s0x390cfd5b347eb62d:0x37205b715389640!8m2!3d28.7040592!4d77.1024902
Error:  name 're' is not defined


# Q6. Write a program to scrap details of all the funding deals for second quarter (i.e Jan 21 – March 21) from trak.in

In [37]:
driver.get('https://trak.in/')

In [38]:
button = driver.find_element_by_xpath('//li[@id="menu-item-51510"]/a').get_attribute('href')
driver.get(button)

<ipython-input-38-fb4fe0726a20>:1: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  button = driver.find_element_by_xpath('//li[@id="menu-item-51510"]/a').get_attribute('href')


In [39]:
fund_dict = {}
fund_dict['Date'] = []
fund_dict['Startup Name'] = []
fund_dict['Industry/Vertical'] = []
fund_dict['Sub-Vertical'] = []
fund_dict['Location'] = []
fund_dict['Investor'] = []
fund_dict['Investment Type'] = []
fund_dict['Amount(in USD)'] = []

In [40]:
for i in range(48,51):
    driver.find_element_by_xpath('//div[@id="tablepress-{}_wrapper"]/div/label/select/option[4]'.format(i)).click()

    # Date
    dt = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[2]'.format(i))
    for d in dt:
        fund_dict['Date'].append(d.text)

    # Startup Name
    sn = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[3]'.format(i))
    for n in sn:
        fund_dict['Startup Name'].append(n.text)
    
    # Industry/Vertical
    ind = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[4]'.format(i))
    for n in ind:
        fund_dict['Industry/Vertical'].append(n.text)
    
    # Sub-Vertical
    sv = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[5]'.format(i))
    for s in sv:
        fund_dict['Sub-Vertical'].append(s.text)

    # Location
    loc = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[6]'.format(i))
    for l in loc:
        fund_dict['Location'].append(l.text)
    
    # Investor
    inv = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[7]'.format(i))
    for n in inv:
        fund_dict['Investor'].append(n.text)
    
    # Investment Type
    invt = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[8]'.format(i))
    for n in invt:
        fund_dict['Investment Type'].append(n.text)
    
    # Amount
    amt = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[9]'.format(i))
    for a in amt:
        fund_dict['Amount(in USD)'].append(a.text)
    
fund_df = pd.DataFrame(fund_dict)
fund_df

<ipython-input-40-829854e7bba2>:2: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  driver.find_element_by_xpath('//div[@id="tablepress-{}_wrapper"]/div/label/select/option[4]'.format(i)).click()
<ipython-input-40-829854e7bba2>:5: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  dt = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[2]'.format(i))
<ipython-input-40-829854e7bba2>:10: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  sn = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[3]'.format(i))
<ipython-input-40-829854e7bba2>:15: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  ind = driver.find_elements_by_xpath('//table[@id="tablepress-{}"]/tbody/tr/td[4]'.format

,Date,Startup Name,Industry/Vertical,Sub-Vertical,Location,Investor,Investment Type,Amount(in USD)
0,15/07/2020,Flipkart,E-commerce,E-commerce,Bangalore,Walmart Inc,M&A,"1,200,000,000"
1,16/07/2020,Vedantu,EduTech,Online Tutoring,Bangalore,Coatue Management,Series D,"100,000,000"
2,16/07/2020,Crio,EduTech,Learning Platform for Developers,Bangalore,021 Capital,pre-Series A,"934,160"
3,14/07/2020,goDutch,FinTech,Group Payments,Mumbai,"Matrix India,Y Combinator, Global Founders Cap...",Seed,"1,700,000"
4,13/07/2020,Mystifly,Airfare Marketplace,"Ticketing, Airline Retailing, and Post-Ticketi...",Singapore and Bangalore,Recruit Co. Ltd.,pre-Series B,"3,300,000"
5,09/07/2020,JetSynthesys,Gaming and Entertainment,Gaming and Entertainment,Pune,Adar Poonawalla and Kris Gopalakrishnan.,Venture-Series Unknown,"400,000"
6,10/07/2020,gigIndia,Marketplace,"Crowd Sourcing, Freelance",Pune,Incubate Fund India and Beyond Next Ventures,pre-Series A,"974,200"
7,15/07/2020,PumPumPum,Automotive Rental,Used Car-leasing platform,Gurgaon,Early Adapters Syndicate,Seed,"292,800"
8,14/07/2020,FLYX,OTT Player,Streaming Social Network,New York and Delhi,"Raj Mishra, founder of AIT Global Inc",pre-Seed,"200,000"
9,13/07/2020,Open Appliances Pvt. Ltd.,Information Technology,Internet-of-Things Security Solutions,Bangalore,Unicorn India Ventures,Venture-Series Unknown,"500,000"


In [42]:
fund_df.to_csv("Indian Startups_Q2_2021.csv")

# Q7. Write a program to scrap all the available details of best gaming laptops from digit.in

In [14]:
# Activating the chrome browser
driver=webdriver.Chrome("chromedriver.exe") 
time.sleep(2)

<ipython-input-14-87d1a30d2b16>:2: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver=webdriver.Chrome("chromedriver.exe")


In [15]:
#Opening the specified url
url = "https://www.digit.in/"
driver.get(url)
time.sleep(3)
#searching for best laptop
best_gam_lap = driver.find_element_by_xpath("//div[@class='listing_container']//ul//li[9]").click()
time.sleep(4)

<ipython-input-15-08a1f1d8b365>:6: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  best_gam_lap = driver.find_element_by_xpath("//div[@class='listing_container']//ul//li[9]").click()


In [16]:
#Creating empty lists
lap_name = []
ope_sys = []
display = []
processor = []
memory = []
weight = []
dimensions = []
graph_proc = []
price = []

In [17]:
# Scraping the data of laptop names
name_tags = driver.find_elements_by_xpath("//table[@id='summtable']//tr//td[1]")
for name in name_tags:
    lap_name.append(name.text)

<ipython-input-17-9131ac4ae0c7>:2: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  name_tags = driver.find_elements_by_xpath("//table[@id='summtable']//tr//td[1]")


In [18]:
#Scraping the data of operating system
try:
    os_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[3]//td[3]")
    for os in os_tags:
        ope_sys.append(os.text)
except NoSuchElementException:
    pass

<ipython-input-18-14efb393a873>:3: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  os_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[3]//td[3]")


In [19]:
#Scraping data of display
try:
    disp_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[4]//td[3]")
    for disp in disp_tags:
        display.append(disp.text)
except NoSuchElementException:
    pass

<ipython-input-19-651ed997d169>:3: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  disp_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[4]//td[3]")


In [20]:
#Scraping data of Processor
try:
    pro_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[5]//td[3]")
    for pro in pro_tags:
        processor.append(pro.text)
except NoSuchElementException:
    pass

<ipython-input-20-eb701594dc4e>:3: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  pro_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[5]//td[3]")


In [21]:
#Scraping data of memory
try:
    memo_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[6]//td[3]")
    for memo in memo_tags:
        memory.append(memo.text)
except NoSuchElementException:
    pass

<ipython-input-21-095f9a671b6f>:3: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  memo_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[6]//td[3]")


In [22]:
#Scraping data of weight
try:
    wgt_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[7]//td[3]")
    for wgt in wgt_tags:
        weight.append(wgt.text)
except NoSuchElementException:
    pass

<ipython-input-22-f96d7229ad0c>:3: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  wgt_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[7]//td[3]")


In [23]:
#Scraping data of dimensions
try:
    dim_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[8]//td[3]")
    for dim in dim_tags:
        dimensions.append(dim.text)
except NoSuchElementException:
    pass

<ipython-input-23-8186d7a01108>:3: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  dim_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[8]//td[3]")


In [24]:
#Scraping data of Graph processor
try:
    gra_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[9]//td[3]")
    for gra in gra_tags:
        graph_proc.append(gra.text)
except NoSuchElementException:
    pass

<ipython-input-24-cf95282c2057>:3: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  gra_tags = driver.find_elements_by_xpath("//div[@class='Spcs-details']//tr[9]//td[3]")


In [25]:
#Scraping data of price
try:
    pri_tags = driver.find_elements_by_xpath("//td[@class='smprice']")
    for pri in pri_tags:
        price.append(pri.text.replace('₹','Rs '))
except NoSuchElementException:
    pass

<ipython-input-25-ba9b948188fc>:3: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  pri_tags = driver.find_elements_by_xpath("//td[@class='smprice']")


In [26]:
#Creating Data Frame
Gaming_Laptop=pd.DataFrame({})
Gaming_Laptop['Laptop Name'] = lap_name
Gaming_Laptop['Operating system'] = ope_sys
Gaming_Laptop['Display'] = display
Gaming_Laptop['Processor'] = processor
Gaming_Laptop['Memory'] = memory
Gaming_Laptop['Weight'] = weight
Gaming_Laptop['Dimensions'] = dimensions
Gaming_Laptop['Graphical Processor'] = graph_proc
Gaming_Laptop['Price'] = price

In [27]:
#Printing data frame
Gaming_Laptop

,Laptop Name,Operating system,Display,Processor,Memory,Weight,Dimensions,Graphical Processor,Price
0,MSI Raider GE76,Windows 11 Home,"17"" (3840 x 2160)",12th Gen Intel Core i9-12900HK | 5 GHz,2 TB SSD/16 GBGB DDR5,2.9,397 x 284 x 26,NVIDIA GeForce RTX 3080Ti,"Rs 429,940"
1,ASUS ROG Strix Scar 15,Windows 11 Home,"15.6"" (2560 x 1440)",12th Gen Intel Core i9-12900H | 2.5 GHz,2 TB SSD/32 GBGB DDR5,2.3,259 x 354 x 27,NVIDIA GeForce RTX 3070 Ti,"Rs 285,390"
2,Acer Nitro 5,Windows 10,"15.6"" (1920 x 1080)",AMD Ryzen 9 Octa Core | 2.4 GHz,1 TB HDD/16 GBGB DDR4,2.4,363.4 x 255 x 23.9,NVIDIA GeForce RTX 3070,"Rs 129,990"
3,MSI Stealth 15M,Windows 10,"15.6"" (1920 x 1080)",Intel Core i7 11th Gen - 11375H | NA,1 TB SSD/16 GBGB DDR4,1.7,358.3 x 248 x 16.15,NVIDIA GeForce RTX 3060,"Rs 134,990"
4,ASUS ROG Strix Scar 15,Windows 10,"15.6"" (2560 x 1440)",AMD Ryzen 9 Octa Core - 5900HX | 3.3 GHz,2 TB SSD/32 GBGB DDR4,2.30,354 x 259 x 22.6,NVIDIA GeForce RTX 3080,"Rs 193,990"
5,ASUS ROG Strix Scar 15,Windows 10 Home,"15.6"" (1920 x 1080)",AMD Ryzen™ 9 5900HX | 3.3 GHz,1 TB SSD/16 GBGB DDR4,2.30,35.4 x 25.9 x 2.26,NVIDIA® GeForce RTX™ 3070,"Rs 215,990"
6,ASUS ZEPHYRUS G14,Windows 10 Home,"14"" (1920 x 1080)",AMD 3rd Gen Ryzen 9 | 3.3 GHz,1 TB SSD/16 GBGB DDR4,1.65,32.5 x 22.1 x 1.8,NVIDIA GeForce RTX 2060,"Rs 144,990"
7,HP Omen 16,Windows 11 Home,"16.1"" (1920 x 1080)",12th Gen Intel Core i7-12700H | 4.7 GHz,1 TB SSD/16 GBGB DDR4,2.32,36.92 x 24.8 x 2.3,NVIDIA GeForce RTX 3060,"Rs 139,990"
8,ASUS ROG ZEPHYRUS DUO 15,Windows 10,"15.6"" (3840 x 1100)",Intel Core i7 10th Gen 10875H | NA,512 GB SSD/4 GBGB DDR4,2.4,268.30 x 360.00 x 20.90,NVIDIA GeForce RTX 2070 Max-Q,"Rs 185,000"
9,Acer Aspire 7 gaming laptop,Windows 10 Home,"15.6"" (1920 x 1080)",AMD Ryzen™ 5-5500U hexa-core | NA,512 GB SSD/8 GBGB DDR4,2.15,2.29 x 36.3 x 25.4,NVIDIA® GeForce® GTX 1650,"Rs 53,490"


# Q8. Write a python program to scrape the details for all billionaires from www.forbes.com. Details to be scrapped: “Rank”, “Name”, “Net worth”, “Age”, “Citizenship”, “Source”, “Industry”.

In [50]:
# Activating the chrome browser
driver=webdriver.Chrome("chromedriver.exe") 
time.sleep(2)

#get the specified url
url = "https://www.forbes.com/?sh=69e6b8c92254"
driver.get(url)
time.sleep(3)

#clicking the explore button
button = driver.find_element_by_xpath("//button[@class='icon--hamburger']")
button.click()
time.sleep(1)

#select billionaire  
bill = driver.find_element_by_xpath("/html/body/div[1]/header/nav/div[3]/ul/li[1]")
bill.click()
time.sleep(1)

#select world billionaire  
world_bill= driver.find_element_by_xpath("/html/body/div[1]/header/nav/div[3]/ul/li[1]/div[2]/ul/li[2]/a")
world_bill.click()
time.sleep(1)

<ipython-input-50-c733160fe6ea>:2: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver=webdriver.Chrome("chromedriver.exe")
<ipython-input-50-c733160fe6ea>:11: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  button = driver.find_element_by_xpath("//button[@class='icon--hamburger']")
<ipython-input-50-c733160fe6ea>:16: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  bill = driver.find_element_by_xpath("/html/body/div[1]/header/nav/div[3]/ul/li[1]")
<ipython-input-50-c733160fe6ea>:21: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  world_bill= driver.find_element_by_xpath("/html/body/div[1]/header/nav/div[3]/ul/li[1]/div[2]/ul/li[2]/a")


In [51]:
#Scraping required data from the web page
#Creating empty lists
Rank = []
Person_Name = []
total_net_worth = []
Age = []
citizenship = []
Source = []
industry = []


while(True):
    #Scraping the data of rank
    rank_tags= driver.find_elements_by_xpath("//div[@class='rank']")
    for rank in rank_tags:
        Rank.append(rank.text)
    time.sleep(1)
    
    
    #Scraping the data of names
    name_tags= driver.find_elements_by_xpath("//div[@class='personName']//div")
    for name in name_tags:
        Person_Name.append(name.text)
    time.sleep(1)
    
    
    #Scraping data of age
    age_tags= driver.find_elements_by_xpath("//div[@class='age']//div")
    for age in age_tags:
        Age.append(age.text)   
    time.sleep(1)
    
    
    #Scraping data of citizenship
    cit_tags= driver.find_elements_by_xpath("//div[@class='countryOfCitizenship']")
    for cit in cit_tags:
        citizenship.append(cit.text)
    time.sleep(1)
    
    
    #Scraping data of source of income
    sour_tags= driver.find_elements_by_xpath("//div[@class='source']")
    for sour in sour_tags:
        Source.append(sour.text)    
    time.sleep(1)
    
    
    #Scraping data of Industry
    ind_tags= driver.find_elements_by_xpath("//div[@class='category']//div")
    for ind in ind_tags:
        industry.append(ind.text)
        
        
    #scraping data of net_worth of billionaire 
    net_tags= driver.find_elements_by_xpath("//div[@class='netWorth']//div[1]")
    for net in net_tags:
        total_net_worth.append(net.text)
    time.sleep(1)
    
    
    #Clicking on next button
    try:
        next_button = driver.find_element_by_xpath("//button[@class='pagination-btn pagination-btn--next ']")
        next_button.click()
    except:
        break
        
#Scraping data of net worth
Net_Worth = []
for i in range(0,len(total_net_worth),2):
    Net_Worth.append(total_net_worth[i])

<ipython-input-51-2db941516e87>:14: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  rank_tags= driver.find_elements_by_xpath("//div[@class='rank']")
<ipython-input-51-2db941516e87>:21: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  name_tags= driver.find_elements_by_xpath("//div[@class='personName']//div")
<ipython-input-51-2db941516e87>:28: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  age_tags= driver.find_elements_by_xpath("//div[@class='age']//div")
<ipython-input-51-2db941516e87>:35: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  cit_tags= driver.find_elements_by_xpath("//div[@class='countryOfCitizenship']")
<ipython-input-51-2db941516e87>:42: DeprecationWarning: find_elements_by_xpath is deprecate

In [52]:
#Framing data
Billionaires=pd.DataFrame({})
Billionaires['Rank'] = Rank
Billionaires['Names'] = Person_Name
Billionaires['Net Worth'] = Net_Worth
Billionaires['Age'] = Age
Billionaires['Citizenship'] = citizenship
Billionaires['Source'] = Source
Billionaires['Industry'] = industry

In [53]:
#Printing dataframe
Billionaires

,Rank,Names,Net Worth,Age,Citizenship,Source,Industry
0,1.,Elon Musk,$219 B,50,United States,"Tesla, SpaceX",Automotive
1,2.,Jeff Bezos,$171 B,58,United States,Amazon,Technology
2,3.,Bernard Arnault & family,$158 B,73,France,LVMH,Fashion & Retail
3,4.,Bill Gates,$129 B,66,United States,Microsoft,Technology
4,5.,Warren Buffett,$118 B,91,United States,Berkshire Hathaway,Finance & Investments
...,...,...,...,...,...,...,...
195,192.,Marcel Herrmann Telles,$10.3 B,72,Brazil,beer,Food & Beverage
196,197.,Leon Black,$10 B,70,United States,private equity,Finance & Investments
197,197.,Joe Gebbia,$10 B,40,United States,Airbnb,Technology
198,197.,David Geffen,$10 B,79,United States,"movies, record labels",Media & Entertainment


# Q9. Write a program to extract at least 500 Comments, Comment upvote and time when comment was posted from any YouTube Video.

In [54]:
# Activating the chrome browser
driver=webdriver.Chrome("chromedriver.exe")
time.sleep(2)

<ipython-input-54-3f35d0d97847>:2: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver=webdriver.Chrome("chromedriver.exe")


In [55]:
# Opening youtube
url = "https://www.youtube.com/"
driver.get(url)
time.sleep(2)

In [56]:
#finding element for search bar
search_bar = driver.find_element_by_id('search')
search_bar.send_keys("GOT")  #entering Video name
time.sleep(1)

<ipython-input-56-3f0f621ccdca>:2: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  search_bar = driver.find_element_by_id('search')


ElementNotInteractableException: Message: element not interactable
  (Session info: chrome=101.0.4951.67)
Stacktrace:
Backtrace:
	Ordinal0 [0x00B07413+2389011]
	Ordinal0 [0x00A99F61+1941345]
	Ordinal0 [0x0098C520+836896]
	Ordinal0 [0x009B48E3+1001699]
	Ordinal0 [0x009B3FBE+999358]
	Ordinal0 [0x009D414C+1130828]
	Ordinal0 [0x009AF974+981364]
	Ordinal0 [0x009D4364+1131364]
	Ordinal0 [0x009E4302+1196802]
	Ordinal0 [0x009D3F66+1130342]
	Ordinal0 [0x009AE546+976198]
	Ordinal0 [0x009AF456+980054]
	GetHandleVerifier [0x00CB9632+1727522]
	GetHandleVerifier [0x00D6BA4D+2457661]
	GetHandleVerifier [0x00B9EB81+569713]
	GetHandleVerifier [0x00B9DD76+566118]
	Ordinal0 [0x00AA0B2B+1968939]
	Ordinal0 [0x00AA5988+1989000]
	Ordinal0 [0x00AA5A75+1989237]
	Ordinal0 [0x00AAECB1+2026673]
	BaseThreadInitThunk [0x7755FA29+25]
	RtlGetAppContainerNamedObjectPath [0x77E97A7E+286]
	RtlGetAppContainerNamedObjectPath [0x77E97A4E+238]


In [57]:
#clicking on search button
search_btn = driver.find_element_by_id("search-icon-legacy")  
search_btn.click()
time.sleep(1)

<ipython-input-57-cac8bc6eb7b2>:2: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  search_btn = driver.find_element_by_id("search-icon-legacy")


In [67]:
#clicking on first video
link_click = driver.find_element_by_xpath("//a[@class='style-scope ytd-video-renderer']")
link_click.click()

<ipython-input-67-68cbc38f8fda>:2: DeprecationWarning: find_element_by_xpath is deprecated. Please use find_element(by=By.XPATH, value=xpath) instead
  link_click = driver.find_element_by_xpath("//a[@class='style-scope ytd-video-renderer']")


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//a[@class='style-scope ytd-video-renderer']"}
  (Session info: chrome=101.0.4951.67)
Stacktrace:
Backtrace:
	Ordinal0 [0x00B07413+2389011]
	Ordinal0 [0x00A99F61+1941345]
	Ordinal0 [0x0098C658+837208]
	Ordinal0 [0x009B91DD+1020381]
	Ordinal0 [0x009B949B+1021083]
	Ordinal0 [0x009E6032+1204274]
	Ordinal0 [0x009D4194+1130900]
	Ordinal0 [0x009E4302+1196802]
	Ordinal0 [0x009D3F66+1130342]
	Ordinal0 [0x009AE546+976198]
	Ordinal0 [0x009AF456+980054]
	GetHandleVerifier [0x00CB9632+1727522]
	GetHandleVerifier [0x00D6BA4D+2457661]
	GetHandleVerifier [0x00B9EB81+569713]
	GetHandleVerifier [0x00B9DD76+566118]
	Ordinal0 [0x00AA0B2B+1968939]
	Ordinal0 [0x00AA5988+1989000]
	Ordinal0 [0x00AA5A75+1989237]
	Ordinal0 [0x00AAECB1+2026673]
	BaseThreadInitThunk [0x7755FA29+25]
	RtlGetAppContainerNamedObjectPath [0x77E97A7E+286]
	RtlGetAppContainerNamedObjectPath [0x77E97A4E+238]


In [68]:
# 1000 time we scroll down to generate more Comments
for _ in range(1000):
    driver.execute_script("window.scrollBy(0,10000)")

In [69]:
#make empty lists
comments = []
comment_time = []
Time = []
Likes = []
No_of_Likes = []

In [70]:
#scraping the data of comments
cm_tags = driver.find_elements_by_id("content-text")
for cm in cm_tags:
    if cm.text is None:
        comments.append("--")
    else:
        comments.append(cm.text)
time.sleep(5)

<ipython-input-70-8a8db7b5202d>:2: DeprecationWarning: find_elements_by_id is deprecated. Please use find_elements(by=By.ID, value=id_) instead
  cm_tags = driver.find_elements_by_id("content-text")


In [71]:
#Creating dataframe
Youtube=pd.DataFrame({})
Youtube['Comments'] = comments
Youtube['Comment_time'] = comment_time
Youtube['Comment upvotes'] = No_of_Likes

In [72]:
#Printing dataframe
Youtube

,Comments,Comment_time,Comment upvotes


# Q10. Write a python program to scrape a data for all available Hostels from https://www.hostelworld.com/ in “London” location. You have to scrape hostel name, distance from city centre, ratings, total reviews, overall reviews, privates from price, dorms from price, facilities and property description.

In [73]:
# Activating the chrome browser
driver=webdriver.Chrome("chromedriver.exe") 
time.sleep(3)

#get the web page with given url
url = "https://www.hostelworld.com/"
driver.get(url)
time.sleep(5)

<ipython-input-73-443cffd4bca9>:2: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver=webdriver.Chrome("chromedriver.exe")


In [74]:
#locating the location search bar
search_loc = driver.find_element_by_id('search-input-field')
# write Lonodn in search bar
search_loc.send_keys("London")
time.sleep(5)

#select london
london = driver.find_element_by_xpath('/html/body/div[1]/div/div/div[1]/div[1]/div/div[2]/div[4]/div/div[2]/div/div[1]/div/div/ul/li[2]/div')
london.click()
time.sleep(5)


# do click on search button
search_btn = driver.find_element_by_id('search-button')
search_btn.click()

<ipython-input-74-bc5bf669500f>:2: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  search_loc = driver.find_element_by_id('search-input-field')


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="search-input-field"]"}
  (Session info: chrome=101.0.4951.67)
Stacktrace:
Backtrace:
	Ordinal0 [0x00B07413+2389011]
	Ordinal0 [0x00A99F61+1941345]
	Ordinal0 [0x0098C658+837208]
	Ordinal0 [0x009B91DD+1020381]
	Ordinal0 [0x009B949B+1021083]
	Ordinal0 [0x009E6032+1204274]
	Ordinal0 [0x009D4194+1130900]
	Ordinal0 [0x009E4302+1196802]
	Ordinal0 [0x009D3F66+1130342]
	Ordinal0 [0x009AE546+976198]
	Ordinal0 [0x009AF456+980054]
	GetHandleVerifier [0x00CB9632+1727522]
	GetHandleVerifier [0x00D6BA4D+2457661]
	GetHandleVerifier [0x00B9EB81+569713]
	GetHandleVerifier [0x00B9DD76+566118]
	Ordinal0 [0x00AA0B2B+1968939]
	Ordinal0 [0x00AA5988+1989000]
	Ordinal0 [0x00AA5A75+1989237]
	Ordinal0 [0x00AAECB1+2026673]
	BaseThreadInitThunk [0x7755FA29+25]
	RtlGetAppContainerNamedObjectPath [0x77E97A7E+286]
	RtlGetAppContainerNamedObjectPath [0x77E97A4E+238]


In [75]:
hostel_name = []
distance = []
pvt_prices = []
dorms_price = []
rating = []
reviews = []
over_all = []
facilities = []
description =[]
product_url = []

In [76]:
#Scraping the requered informations
for i in driver.find_elements_by_xpath("//div[@class = 'pagination-item pagination-current' or @class='pagination-item']"):
    i.click()
    time.sleep(4)
    #fetching hostel name
    try:
        name = driver.find_elements_by_xpath("//h2[@class='title title-6']")
        for i in name:
            hostel_name.append(i.text)
    except NoSuchElementException:
        hostel_name.append('-')
    #fetching distance from city centre
    
    try:
        dist = driver.find_elements_by_xpath("//div[@class='subtitle body-3']//a//span[1]")
        for i in dist:
            distance.append(i.text.replace('Hostel - ',''))
    except NoSuchElementException:
        distance.append('-')
        
    for i in driver.find_elements_by_xpath("//div[@class='prices-col']"):
    #fetch privates from price
        try:
            pvt_price = driver.find_element_by_xpath("//a[@class='prices']//div[1]//div")
            pvt_prices.append(pvt_price.text)
        except NoSuchElementException:
            pvt_prices.append('-')
    #fetching dorms from price
    for i in driver.find_elements_by_xpath("//div[@class='prices-col']"):
        try:
            dorms = driver.find_element_by_xpath("//a[@class='prices']//div[2]//div")
            dorms_price.append(dorms.text)
        except NoSuchElementException:
            dorms_price.append('-')
            #fetching facilities
    try:
        fac1 = driver.find_elements_by_xpath("//div[@class='has-wifi']")
        fac2 = driver.find_elements_by_xpath("//div[@class='has-sanitation']")
        for i in fac1:
            for j in fac2:
                facilities.append(i.text +', '+ j.text )
    except NoSuchElementException:
        facilities.append('-')
    #lets fetch url of each hostel
    p_url = driver.find_elements_by_xpath("//div[@class='prices-col']//a[2]")
    for i in p_url:
        product_url.append(i.get_attribute('href'))

for i in product_url:
    driver.get(i)
    time.sleep(3)
    #lets click on show more button for description
    try:
        driver.find_element_by_xpath("//a[@class='toggle-content']").click()
        time.sleep(5)
    except NoSuchElementException:
        pass
    #fetching ratings
    try:
        rat = driver.find_element_by_xpath("//div[@class='score orange big' or @class='score gray big']")
        rating.append(rat.text)
    except NoSuchElementException:
        rating.append('-')
    #fetching total reviews
        
    try:
        rws = driver.find_element_by_xpath("//div[@class='reviews']")
        reviews.append(rws.text.replace('Total Reviews',''))
    except NoSuchElementException:
        reviews.append('-')
        #fetch overall review
    try:
        overall_rw = driver.find_element_by_xpath("//div[@class='keyword']//span")
        over_all.append(overall_rw.text)
    except NoSuchElementException:
        over_all.append('-')
    #fetch property description 
    try:
        disc = driver.find_element_by_xpath("//div[@class='content']")
        description.append(disc.text)
    except NoSuchElementException:
        over_all.append('-')

<ipython-input-76-27a01b06201f>:2: DeprecationWarning: find_elements_by_xpath is deprecated. Please use find_elements(by=By.XPATH, value=xpath) instead
  for i in driver.find_elements_by_xpath("//div[@class = 'pagination-item pagination-current' or @class='pagination-item']"):


In [77]:
#Creating dataframe
dff = pd.DataFrame({})
dff['Hostel_Name'] = hostel_name
dff['Distance fron city centre'] = distance
dff['Ratings'] = rating
dff['Total_reviews'] = reviews
dff['Overall Reviews'] = over_all
dff['Privates from price'] = pvt_prices
dff['Dorms from price'] = dorms_price
dff['Facilities'] = facilities[:83]
dff['Description'] = description

In [78]:
#Printing data frame
dff

,Hostel_Name,Distance fron city centre,Ratings,Total_reviews,Overall Reviews,Privates from price,Dorms from price,Facilities,Description
